In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt

## Comparing the exciton density

`n_optim` is dencity on the optimized geometry.
`n_orig` — is dencity on the non-optimized geometry.

In [ ]:
def load_first(path):
    d = np.load(path)
    return d[next(iter(d.keys()))]

n_optim = load_first('n_optim.npz')
n_orig  = load_first('n_orig.npz')

max_optim = n_optim.max()
max_orig  = n_orig.max()

print(f"n_optim : max = {max_optim:.4e}")
print(f"n_orig : max = {max_orig:.4e}")
print()
print(f"optim / orig = {max_optim / max_orig:.4f}")
print(f"orig / optim = {max_orig  / max_optim:.4f}")
print(f"Δ (optim - orig) = {max_optim - max_orig:.4e}")
print(f"Δ / orig = {(max_optim - max_orig) / max_orig * 100:.2f} %")

## Support map 

In [ ]:
NPZ_FILENAME   = "swg_256.npz"
NPZ_ARRAY_NAME = 'array'
width  = 12e-6
height = 12e-6

support_data = np.load(NPZ_FILENAME)
if NPZ_ARRAY_NAME in support_data:
    support_map = support_data[NPZ_ARRAY_NAME]
else:
    support_map = support_data[support_data.files[0]]

support_map = np.where(support_map > 0.5, 1, 0)

Ny, Nx = support_map.shape
dx = width  / (Nx - 1)
dy = height / (Ny - 1)

x_coords = np.linspace(0, width,  Nx)
y_coords = np.linspace(0, height, Ny)
X, Y = np.meshgrid(x_coords, y_coords)

## E-field from FDTD


In [ ]:
matrix = np.loadtxt('TE_field_plenka.txt')
E_map = matrix / np.max(matrix)

## 6. Interpolating the field on the exciton density grid

In [ ]:
L_em = 12e-6
L_n  = 12e-6

Ny_em, Nx_em = E_map.shape
Ny_n,  Nx_n  = n_orig.shape

x_em = np.linspace(-L_em/2, L_em/2, Nx_em)
y_em = np.linspace(-L_em/2, L_em/2, Ny_em)

x_n = np.linspace(-L_n/2, L_n/2, Nx_n)
y_n = np.linspace(-L_n/2, L_n/2, Ny_n)
X_n, Y_n = np.meshgrid(x_n, y_n)

x0 = 6e-6
y0 = 6.0e-6

interp = RegularGridInterpolator(
    (y_em, x_em),
    E_map,
    bounds_error=False,
    fill_value=0.0
)

points = np.stack([
    (Y - y0).ravel(),
    (X - x0).ravel()
], axis=-1)

E_on_n = interp(points).reshape(Ny_n, Nx_n)

## Interpolation check

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(
    np.rot90(E_on_n),
    origin='lower',
    extent=[X.min()*1e6, X.max()*1e6, Y.min()*1e6, Y.max()*1e6],
    cmap='hot',
    aspect='equal'
)
plt.colorbar()
plt.xlabel('X, microns')
plt.ylabel('Y, micrins')
plt.title('Interpolated E-field')
plt.show()

## Overlap integral



In [ ]:
E_on_n = np.rot90(E_on_n)

n_optim_normed = n_optim / np.max(n_optim)
n_orig_normed  = n_orig  / np.max(n_optim)

E_on_n_normed = E_on_n / np.max(E_on_n)

overlap_optim = np.sum(np.abs(E_on_n_normed)**2 * n_optim_normed)
overlap_orig  = np.sum(np.abs(E_on_n_normed)**2 * n_orig_normed)

print(f"Overlap (optimized SWG) = {overlap_optim:.4e}")
print(f"Overlap (original SWG)  = {overlap_orig:.4e}")
print(f"Relative increase = {(overlap_optim / overlap_orig - 1) * 100:.2f} %")